# V2 Seed Addendum - Disclosed Post-Hoc Official-Validation Seed Addendum

Research question: how dispersed is the frozen primary spec's official-validation
margin over the same-row stratified dummy across six additional seeds?

Scope: `validation_only`. This notebook trains the FROZEN Stage 02 primary
(`price_volume_time_w20` / `tcn` / `tcn_p01`) under six additional seeds and scores
each seed EXACTLY ONCE on the official validation split (2013-09-16 to 2017-01-25).

**Hard rules.** The addendum NEVER reruns the predeclared seeds 101/202, NEVER touches
rows at or after the closed holdout boundary (2017-01-25; asserted in code), selects
nothing, judges nothing pass/fail, and is NEVER merged into the predeclared n=2
readout - which remains the paper's canonical official-validation number either way.

## Preregistration Summary (frozen before any addendum scoring event)

- Preregistration: `docs/protocols/v2_seed_addendum_preregistration_20260701.md`
- Seed rule (deterministic, declared): `seed_k = 101 * k`; the predeclared readout used
  `k in {1, 2}` -> `[101, 202]`; the addendum takes the next six consecutive multipliers
  `k in {3..8}` -> `[303, 404, 505, 606, 707, 808]`.
- Pipeline per seed: frozen Stage 00 data path -> frozen Stage 01 candidate windows
  (Stage 01 parity gate) -> mechanism-frozen tcn_tiny refit on all eligible official-train
  rows (chronological-tail early stopping, identical Stage 02/03 torch defaults) -> ONE
  scoring pass on all eligible official-validation rows vs the four same-row registry
  baselines (per-seed stratified-dummy draws, Stage 03 convention).
- Predeclared reporting: ALL six seeds reported regardless of outcome; summary =
  min/median/max delta vs stratified dummy + count of seeds with positive delta;
  interpretation is descriptive dispersion evidence only. A wide or negative-touching
  spread is reported honestly and would strengthen the paper's stated n=2 limitation.
- Budget: exactly 6 new official-validation scoring contacts,
  `contact_type=official_validation_seed_addendum`, `for_selection=false`, each logged in
  the scoring-event ledger and emitted as an appendable Stage 05 budget-ledger row.

## Expected Artifacts

Written to `/content/lst_models_results/v2_seed_addendum_readout/<run_id>/`, then backed
up to `My Drive/lst_models/results/v2_seed_addendum_readout/<run_id>/`:

```text
v2sa_validation_readout.csv       per-seed rows, Stage 03 readout schema
v2sa_per_ticker_readout.csv       per-ticker rows per seed, Stage 03 schema
v2sa_seed_dispersion_summary.csv  predeclared min/median/max + positive-seed counts
v2sa_same_row_baselines.csv       four registry baselines per seed, same rows
v2sa_validation_predictions.csv   per-row prediction dump (measure-only reuse later)
v2sa_budget_ledger_row.csv        exact appendable Stage 05 budget-ledger row
v2sa_decision_record.json         scoring-event ledger + per-seed outcomes
run_manifest.json                 device, package versions, config hash, timestamp
artifact_inventory.csv            bytes + sha256 per artifact
drive_backup_manifest.json        written by the durable-save cell
```

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import subprocess
import sys
import threading
import time
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "f15bc0d05d3f9253423b5524c2393d34659d64d8"
PROJECT_ROOT = Path("/content/lst_models")

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_V2SA_CHECKPOINT_MIRROR = True
RUN_V2SA_DRIVE_BACKUP = True
RUN_V2SA = False
RUN_ARTIFACT_DISPLAY = False

# Exact-run resume: fill ONLY to complete a crashed addendum run. A resumed run
# skips every recorded scoring event; a fresh run keeps this empty.
RESUME_V2SA_RUN_ID = ""
RESUME_V2SA_CHECKPOINT_DIR = ""  # defaults to CHECKPOINT_ROOT / RESUME_V2SA_RUN_ID

STAGE_NAME = "v2_seed_addendum_readout"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False
OFFICIAL_VALIDATION_CONTACT = True
OFFICIAL_VALIDATION_FOR_SELECTION = False
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_RUN_ID = "20260610_082130_797479"
SUPERSEDED_STAGE02_RUN_IDS = ["20260609_100637_704705", "20260610_010019_507648"]
CANONICAL_STAGE03_RUN_ID = "20260610_133305_716174"  # context only; NEVER re-run here
ADDENDUM_SEEDS = [303, 404, 505, 606, 707, 808]  # seed_k = 101*k, k in 3..8
PREDECLARED_SEEDS = [101, 202]  # never rerun by this notebook
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_RUN_ID
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_RUN_ID]
OUTPUT_DIR = Path("/content/lst_models_results/v2_seed_addendum_readout")
V2SA_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_seed_addendum_readout"]
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/v2_seed_addendum_readout")
CHECKPOINT_DRIVE_BASE_PARTS = ["lst_models", "checkpoints", "v2_seed_addendum_readout"]
REQUIRED_V2SA_OUTPUTS = [
    "v2sa_validation_readout.csv",
    "v2sa_per_ticker_readout.csv",
    "v2sa_seed_dispersion_summary.csv",
    "v2sa_same_row_baselines.csv",
    "v2sa_validation_predictions.csv",
    "v2sa_budget_ledger_row.csv",
    "v2sa_decision_record.json",
    "run_manifest.json",
    "artifact_inventory.csv",
]

assert ADDENDUM_SEEDS == [101 * k for k in range(3, 9)], "seed rule drifted"
assert not set(ADDENDUM_SEEDS) & set(PREDECLARED_SEEDS), "addendum must never rerun 101/202"
if STAGE02_RUN_ID in SUPERSEDED_STAGE02_RUN_IDS:
    raise ValueError(f"superseded Stage 02 run id must not be pinned: {STAGE02_RUN_ID}")

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "v2_seed_addendum_readout.yaml").exists()
        and (path / "docs" / "protocols" / "v2_seed_addendum_preregistration_20260701.md").exists()
        and (path / "notebooks" / "v2_seed_addendum_readout_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "v2_seed_addendum_readout.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "<" in PROJECT_REPO_COMMIT:
            raise ValueError("fill PROJECT_REPO_COMMIT with the v2 seed addendum full-bundle commit before bootstrapping")
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_seed_addendum_readout.yaml"
STAGE03_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "03_frozen_validation_readout.yaml"
PREREG_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_seed_addendum_preregistration_20260701.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "v2_seed_addendum_readout_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "v2_seed_addendum_readout.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
REQUIRED_PROJECT_FILES = [
    STAGE_CONFIG_PATH, STAGE03_CONFIG_PATH, PREREG_PATH, NOTEBOOK_PATH,
    STAGE_ENTRYPOINT_PATH, RAW_DATA_MANIFEST_PATH,
]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("v2 seed addendum bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def clear_project_import_cache():
    cached = [name for name in sys.modules if name == "lst_models" or name.startswith("lst_models.")]
    for name in cached:
        del sys.modules[name]
    importlib.invalidate_caches()

clear_project_import_cache()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_BOOTSTRAP_MODE:", PROJECT_BOOTSTRAP_MODE)
print("PROJECT_COMMIT:", PROJECT_REPO_COMMIT)
print("STAGE00_RUN_ID:", STAGE00_RUN_ID)
print("STAGE01_RUN_ID:", STAGE01_RUN_ID)
print("STAGE02_RUN_ID:", STAGE02_RUN_ID)
print("CANONICAL_STAGE03_RUN_ID (context only):", CANONICAL_STAGE03_RUN_ID)
print("ADDENDUM_SEEDS:", ADDENDUM_SEEDS)
print("PREDECLARED_SEEDS (never rerun):", PREDECLARED_SEEDS)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("RUN_V2SA:", RUN_V2SA)

## Config Load, Runtime Injection, And Contract Check

In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the v2 seed addendum config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required v2 seed addendum file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(STAGE03_CONFIG_PATH)
require_path(PREREG_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    v2sa_config = yaml.safe_load(handle)
with STAGE03_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage03_config_reference = yaml.safe_load(handle)

stage_inputs = v2sa_config["inputs"]
stage_outputs = v2sa_config["outputs"]
stage_checkpointing = v2sa_config["checkpointing"]
stage_inputs["stage00_run_id"] = STAGE00_RUN_ID
stage_inputs["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
stage_inputs["stage00_drive_path_parts"] = STAGE00_DRIVE_PATH_PARTS
stage_inputs["stage01_run_id"] = STAGE01_RUN_ID
stage_inputs["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
stage_inputs["stage01_drive_path_parts"] = STAGE01_DRIVE_PATH_PARTS
stage_inputs["stage02_run_id"] = STAGE02_RUN_ID
stage_inputs["stage02_runtime_run_dir"] = str(STAGE02_OUTPUT_DIR)
stage_inputs["stage02_drive_path_parts"] = STAGE02_DRIVE_PATH_PARTS
stage_inputs["superseded_stage02_run_ids"] = SUPERSEDED_STAGE02_RUN_IDS
stage_inputs["raw_data_dir"] = str(RAW_DATA_DIR)
stage_outputs["output_dir"] = str(OUTPUT_DIR)
stage_checkpointing["checkpoint_dir"] = str(CHECKPOINT_ROOT)
if RESUME_V2SA_RUN_ID:
    resume_checkpoint_dir = RESUME_V2SA_CHECKPOINT_DIR or str(CHECKPOINT_ROOT / RESUME_V2SA_RUN_ID)
    v2sa_config["resume"] = {
        "enabled": True,
        "run_id": RESUME_V2SA_RUN_ID,
        "checkpoint_dir": resume_checkpoint_dir,
    }
    print("RESUME MODE: continuing run", RESUME_V2SA_RUN_ID, "from", resume_checkpoint_dir)

assert v2sa_config["stage_name"] == STAGE_NAME
assert v2sa_config["route"] == ROUTE
assert v2sa_config["scope"] == SCOPE
assert v2sa_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert v2sa_config["official_validation_contact"] is OFFICIAL_VALIDATION_CONTACT
assert v2sa_config["official_validation_for_selection"] is OFFICIAL_VALIDATION_FOR_SELECTION
assert v2sa_config["addendum_never_merged_into_predeclared_readout"] is True
assert v2sa_config["evidence_status"] == "official_validation_seed_addendum_disclosed_post_hoc"
readout = v2sa_config["readout"]
assert readout["seed_rule"] == "seed_k_equals_101_times_k_next_six_multipliers_k3_to_k8"
assert readout["addendum_seeds"] == ADDENDUM_SEEDS
assert readout["predeclared_seeds_never_rerun"] == PREDECLARED_SEEDS
assert readout["score_each_seed_candidate_exactly_once"] is True
assert readout["refit_rows"] == "all_eligible_official_train_rows"
assert readout["scoring_rows"] == "all_eligible_official_validation_rows"
assert int(readout["max_materialized_train_bytes"]) > 0
identity = v2sa_config["frozen_primary_identity"]
assert identity["candidate_id"] == "price_volume_time_w20"
assert identity["feature_set"] == "price_volume_time"
assert identity["window_size"] == 20
assert identity["model_family"] == "tcn"
assert identity["probe_id"] == "tcn_tiny"
assert identity["hpo_profile_id"] == "tcn_p01"
# The refit mechanism must be byte-equal to the frozen Stage 03 torch defaults.
assert v2sa_config["probe_training_defaults"]["torch"] == stage03_config_reference["probe_training_defaults"]["torch"]
bounds = v2sa_config["date_bounds"]
assert bounds["closed_holdout_test_start"] == "2017-01-25"
assert bounds["validation_end_exclusive"] == "2017-01-25"
budget = v2sa_config["budget"]
assert int(budget["max_new_official_validation_scoring_events"]) == len(ADDENDUM_SEEDS) == 6
assert budget["contact_type"] == "official_validation_seed_addendum"
assert budget["for_selection"] is False
reporting = v2sa_config["predeclared_reporting"]
assert reporting["report_all_addendum_seeds_regardless_of_outcome"] is True
assert reporting["interpretation"] == "descriptive_dispersion_evidence_only"
assert reporting["canonical_readout"] == "predeclared_n2_seeds_101_202_remains_canonical"
assert v2sa_config["baseline_controls"]["mandatory"] == ["stratified_dummy_train_prior", "majority_train_prior", "constant_up", "constant_down"]
assert stage_inputs["raw_data_manifest"] == "configs/lst_models_data.yaml"
assert stage_inputs["canonical_stage03_run_id"] == CANONICAL_STAGE03_RUN_ID
expected_output_names = {
    "validation_readout": "v2sa_validation_readout.csv",
    "per_ticker_readout": "v2sa_per_ticker_readout.csv",
    "seed_dispersion_summary": "v2sa_seed_dispersion_summary.csv",
    "same_row_baselines": "v2sa_same_row_baselines.csv",
    "validation_predictions": "v2sa_validation_predictions.csv",
    "decision_record": "v2sa_decision_record.json",
    "budget_ledger_row": "v2sa_budget_ledger_row.csv",
    "manifest": "run_manifest.json",
    "artifact_inventory": "artifact_inventory.csv",
}
for output_key, output_name in expected_output_names.items():
    assert stage_outputs[output_key] == output_name

print(json.dumps({
    "stage_name": v2sa_config["stage_name"],
    "addendum_seeds": readout["addendum_seeds"],
    "predeclared_seeds_never_rerun": readout["predeclared_seeds_never_rerun"],
    "frozen_primary_identity": dict(identity),
    "budget_max_new_scoring_events": budget["max_new_official_validation_scoring_events"],
    "official_validation_for_selection": v2sa_config["official_validation_for_selection"],
    "holdout_test_contact": v2sa_config["holdout_test_contact"],
}, indent=2))

## Upstream Inputs: Stage 00, Stage 01, Stage 02, And Raw Data (Drive file IDs)

In [ ]:
from lst_models.artifacts import require_artifacts

def get_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive access only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive", fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def ensure_drive_folder(service, parent_id, name):
    folder_mime = "application/vnd.google-apps.folder"
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false and mimeType = '{folder_mime}'"
    matches = service.files().list(q=query, fields="files(id, name, webViewLink)", pageSize=10).execute().get("files", [])
    if len(matches) == 1:
        return matches[0]["id"]
    if len(matches) > 1:
        raise RuntimeError(f"Duplicate Drive folders named {name!r} under parent {parent_id}")
    created = service.files().create(body={"name": name, "mimeType": folder_mime, "parents": [parent_id]}, fields="id, name, webViewLink").execute()
    return created["id"]

def ensure_drive_path(service, path_parts):
    folder_id = "root"
    for part in path_parts:
        folder_id = ensure_drive_folder(service, folder_id, part)
    return folder_id

def find_drive_children(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false"
    return service.files().list(q=query, fields="files(id, name, mimeType, size, webViewLink)", pageSize=10).execute().get("files", [])

def upload_or_update_drive_file(service, parent_id, local_path, display_name=None):
    from googleapiclient.http import MediaFileUpload
    name = display_name or local_path.name
    matches = find_drive_children(service, parent_id, name)
    media = MediaFileUpload(str(local_path), resumable=True)
    if len(matches) == 0:
        uploaded = service.files().create(body={"name": name, "parents": [parent_id]}, media_body=media, fields="id, name, size, webViewLink").execute()
        action = "uploaded"
    elif len(matches) == 1:
        uploaded = service.files().update(fileId=matches[0]["id"], media_body=media, fields="id, name, size, webViewLink").execute()
        action = "updated"
    else:
        raise RuntimeError(f"Duplicate Drive files named {name!r} under parent {parent_id}")
    print(f"{action}: {name}")
    return dict(uploaded)

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    raw_source = raw_manifest["raw_source"]
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_source["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

required_stage00_artifacts = stage_inputs["required_stage00_artifacts"]
required_stage01_artifacts = stage_inputs["required_stage01_artifacts"]
required_stage02_artifacts = stage_inputs["required_stage02_artifacts"]
if RUN_V2SA:
    service = get_drive_service()
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    missing_raw = [raw_manifest["raw_source"]["files"][ticker]["name"] for ticker in raw_manifest["tickers"] if not (RAW_DATA_DIR / raw_manifest["raw_source"]["files"][ticker]["name"]).exists()]
    if missing_raw and RUN_RAW_DATA_SYNC:
        print("Missing raw data files; syncing from Drive file IDs:", missing_raw)
        sync_raw_data_from_drive(service)
    stage00_missing = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if stage00_missing and RUN_STAGE00_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, stage00_missing, "stage00")
    stage01_missing = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if stage01_missing and RUN_STAGE01_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, stage01_missing, "stage01")
    stage02_missing = [name for name in required_stage02_artifacts if not (STAGE02_OUTPUT_DIR / name).exists()]
    if stage02_missing and RUN_STAGE02_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE02_OUTPUT_DIR, STAGE02_DRIVE_PATH_PARTS, stage02_missing, "stage02")
    require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    require_artifacts(STAGE02_OUTPUT_DIR, required_stage02_artifacts)
    print("Stage 00, Stage 01, Stage 02, and raw data input checks passed.")
else:
    print("RUN_V2SA=False; upstream input sync skipped.")

## GPU Runtime Check And Pre-Flight Feasibility Estimate (Before Any Scoring)

In [ ]:
if RUN_V2SA:
    import pandas as pd
    try:
        import torch
        print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            print("gpu:", torch.cuda.get_device_name(0))
        else:
            print("WARNING: no CUDA GPU visible. The frozen mechanism allows CPU fallback")
            print("(require_gpu=false, recorded in the manifest), but six full-row refits on")
            print("CPU are impractical. Use Runtime > Change runtime type > GPU.")
    except ImportError as exc:
        raise ModuleNotFoundError("torch is required for the tcn_tiny refits") from exc

    handoff_path = STAGE02_OUTPUT_DIR / "02_stage03_handoff.json"
    candidate_inputs_path = STAGE01_OUTPUT_DIR / "01_candidate_inputs.json"
    sample_event_index_path = STAGE00_OUTPUT_DIR / "sample_event_index.csv"
    for path in [handoff_path, candidate_inputs_path, sample_event_index_path]:
        require_path(path)
    stage02_handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
    if stage02_handoff.get("ready_for_stage03") is not True:
        raise RuntimeError("Stage 02 handoff records ready_for_stage03=false; the addendum may only score the frozen readout chain.")
    primary = stage02_handoff["primary_candidate"]
    for key, expected in [("candidate_id", "price_volume_time_w20"), ("model_family", "tcn"), ("hpo_profile_id", "tcn_p01")]:
        if str(primary.get(key)) != expected:
            raise RuntimeError(f"frozen primary identity mismatch on {key}: {primary.get(key)!r} != {expected!r}")
    candidate_inputs = json.loads(candidate_inputs_path.read_text(encoding="utf-8"))["candidate_inputs"]
    matches = [entry for entry in candidate_inputs if str(entry["candidate_id"]) == str(primary["candidate_id"])]
    if len(matches) != 1:
        raise ValueError(f"expected exactly one 01_candidate_inputs.json entry for {primary['candidate_id']!r}; found {len(matches)}")
    events = pd.read_csv(sample_event_index_path)
    valid_label = events["valid_label"].map(lambda value: str(value).strip().lower() in {"true", "1", "yes"})
    n_train_rows = int(((events["split"] == "train") & valid_label).sum())
    n_validation_rows = int(((events["split"] == "validation") & valid_label).sum())
    per_row_bytes = int(primary["window_size"]) * len(matches[0]["feature_columns"]) * 4
    combined_bytes = (n_train_rows + n_validation_rows) * per_row_bytes
    max_bytes = int(v2sa_config["readout"]["max_materialized_train_bytes"])
    print(json.dumps({
        "n_train_rows_upper_bound": n_train_rows,
        "n_validation_rows_upper_bound": n_validation_rows,
        "window_size": int(primary["window_size"]),
        "n_features": len(matches[0]["feature_columns"]),
        "combined_bytes_estimate": combined_bytes,
        "max_materialized_train_bytes": max_bytes,
    }, indent=2))
    if combined_bytes > max_bytes:
        raise RuntimeError(
            "v2 seed addendum pre-flight infeasible BEFORE any scoring event: combined "
            f"estimate {combined_bytes} bytes exceeds max_materialized_train_bytes {max_bytes}; "
            "zero scoring events have occurred"
        )
    print("Pre-flight feasibility passed; proceeding is authorized.")
else:
    print("RUN_V2SA=False; GPU check and pre-flight estimate skipped.")

## Checkpoint Mirror To Drive

In [ ]:
V2SA_MIRROR_STATE = {"stop": False, "errors": [], "uploaded_mtimes": {}, "folder_ids": {}}
V2SA_MIRROR_POLL_SECONDS = 120
V2SA_MIRROR_STABLE_SECONDS = 10

def mirror_v2sa_checkpoints_once(service):
    if not CHECKPOINT_ROOT.exists():
        return 0
    uploaded_count = 0
    now = time.time()
    for path in sorted(CHECKPOINT_ROOT.rglob("*")):
        if not path.is_file():
            continue
        mtime = path.stat().st_mtime
        if now - mtime < V2SA_MIRROR_STABLE_SECONDS:
            continue
        key = str(path)
        if V2SA_MIRROR_STATE["uploaded_mtimes"].get(key) == mtime:
            continue
        relative = path.relative_to(CHECKPOINT_ROOT)
        parent_parts = tuple(CHECKPOINT_DRIVE_BASE_PARTS + list(relative.parent.parts)) if relative.parent != Path(".") else tuple(CHECKPOINT_DRIVE_BASE_PARTS)
        folder_id = V2SA_MIRROR_STATE["folder_ids"].get(parent_parts)
        if folder_id is None:
            folder_id = ensure_drive_path(service, list(parent_parts))
            V2SA_MIRROR_STATE["folder_ids"][parent_parts] = folder_id
        upload_or_update_drive_file(service, folder_id, path)
        V2SA_MIRROR_STATE["uploaded_mtimes"][key] = mtime
        uploaded_count += 1
    return uploaded_count

def v2sa_checkpoint_mirror_loop():
    try:
        mirror_service = get_drive_service()
    except Exception as exc:
        V2SA_MIRROR_STATE["errors"].append(f"mirror auth failed: {type(exc).__name__}: {exc}")
        print("checkpoint mirror disabled:", V2SA_MIRROR_STATE["errors"][-1])
        return
    while not V2SA_MIRROR_STATE["stop"]:
        try:
            mirror_v2sa_checkpoints_once(mirror_service)
        except Exception as exc:
            V2SA_MIRROR_STATE["errors"].append(f"{type(exc).__name__}: {exc}")
            print("checkpoint mirror error (recorded, run continues):", V2SA_MIRROR_STATE["errors"][-1])
        for _ in range(V2SA_MIRROR_POLL_SECONDS):
            if V2SA_MIRROR_STATE["stop"]:
                break
            time.sleep(1)

def start_v2sa_checkpoint_mirror():
    V2SA_MIRROR_STATE["stop"] = False
    thread = threading.Thread(target=v2sa_checkpoint_mirror_loop, name="v2sa_checkpoint_mirror", daemon=True)
    thread.start()
    return thread

def stop_v2sa_checkpoint_mirror(thread):
    V2SA_MIRROR_STATE["stop"] = True
    if thread is not None:
        thread.join(timeout=V2SA_MIRROR_POLL_SECONDS + 30)
    final_service = get_drive_service()
    V2SA_MIRROR_STATE["uploaded_mtimes"] = {}
    uploaded = mirror_v2sa_checkpoints_once(final_service)
    print(f"final checkpoint sweep uploaded {uploaded} files; recorded mirror errors: {len(V2SA_MIRROR_STATE['errors'])}")
    if V2SA_MIRROR_STATE["errors"]:
        print(json.dumps(V2SA_MIRROR_STATE["errors"], indent=2))

RESUME_REQUIRED_CHECKPOINT_FILES = [
    "checkpoint_manifest.json",
    "v2sa_ledger_state_partial.json",
    "v2sa_validation_readout_partial.csv",
    "v2sa_same_row_baselines_partial.csv",
    "v2sa_validation_predictions_partial.csv",
]

def fetch_resume_checkpoint_from_drive():
    """Restore the exact checkpoint folder for RESUME_V2SA_RUN_ID from Drive
    when the local runtime lost it (exact path parts, no scans)."""
    local_dir = Path(RESUME_V2SA_CHECKPOINT_DIR or str(CHECKPOINT_ROOT / RESUME_V2SA_RUN_ID))
    missing = [name for name in RESUME_REQUIRED_CHECKPOINT_FILES if not (local_dir / name).exists()]
    if not missing:
        print("resume checkpoint complete locally:", local_dir)
        return
    service = get_drive_service()
    drive_parts = CHECKPOINT_DRIVE_BASE_PARTS + [RESUME_V2SA_RUN_ID]
    sync_stage_artifacts_from_drive(service, local_dir, drive_parts, missing, "v2sa resume checkpoint")
    still_missing = [name for name in RESUME_REQUIRED_CHECKPOINT_FILES if not (local_dir / name).exists()]
    if still_missing:
        raise FileNotFoundError(f"resume checkpoint files missing after Drive fetch: {still_missing}")

if RUN_V2SA and RESUME_V2SA_RUN_ID:
    fetch_resume_checkpoint_from_drive()

print("Checkpoint mirror helpers ready; RUN_V2SA_CHECKPOINT_MIRROR =", RUN_V2SA_CHECKPOINT_MIRROR)

## Run The Seed Addendum (Six One-Shot Scoring Events)

In [ ]:
if RUN_V2SA:
    try:
        from lst_models.stages.v2_seed_addendum_readout import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Missing entry point: src/lst_models/stages/v2_seed_addendum_readout.py.") from exc
    mirror_thread = start_v2sa_checkpoint_mirror() if RUN_V2SA_CHECKPOINT_MIRROR else None
    try:
        result = run_stage(v2sa_config)
    finally:
        if RUN_V2SA_CHECKPOINT_MIRROR:
            stop_v2sa_checkpoint_mirror(mirror_thread)
    display(result)
    V2SA_RUN_ID = Path(result.output_dir).name
    if Path(result.output_dir).parent != OUTPUT_DIR:
        raise RuntimeError(f"output dir mismatch: {Path(result.output_dir).parent} != {OUTPUT_DIR}")
    run_manifest = json.loads(Path(result.run_manifest).read_text(encoding="utf-8"))
    print("V2SA_RUN_ID:", V2SA_RUN_ID)
    print(json.dumps({
        "run_timestamp_utc": run_manifest.get("run_timestamp_utc"),
        "config_sha256": run_manifest.get("config_sha256"),
        "resolved_device": run_manifest.get("resolved_device"),
        "gpu_name_or_null": run_manifest.get("gpu_name_or_null"),
        "python_version": run_manifest.get("python_version"),
        "package_versions": run_manifest.get("package_versions"),
        "official_validation_scoring_events": run_manifest.get("official_validation_scoring_events"),
    }, indent=2))
else:
    print("RUN_V2SA=False; seed addendum not executed.")

## Durable Drive Result Save

In [ ]:
def backup_v2sa_results_to_drive(output_run_dir):
    run_dir = Path(output_run_dir)
    if not run_dir.exists():
        raise FileNotFoundError(f"v2 seed addendum output folder not found: {run_dir}")
    run_id = run_dir.name
    missing = [name for name in REQUIRED_V2SA_OUTPUTS if not (run_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing required v2 seed addendum artifacts before Drive backup in {run_dir}: {missing}"
        )
    run_manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
    if run_manifest.get("official_validation_for_selection") is not False:
        raise ValueError("backup refused: run manifest must record official_validation_for_selection=false")
    if run_manifest.get("holdout_test_contact") is not False:
        raise ValueError("backup refused: run manifest must record holdout_test_contact=false")
    if run_manifest.get("addendum_never_merged_into_predeclared_readout") is not True:
        raise ValueError("backup refused: run manifest must record addendum_never_merged_into_predeclared_readout=true")
    if run_manifest.get("no_final_model_selected") is not True:
        raise ValueError("backup refused: run manifest must record no_final_model_selected=true")
    if int(run_manifest.get("official_validation_scoring_events", -1)) > 6:
        raise ValueError("backup refused: scoring events exceed the predeclared budget of 6")
    if run_manifest.get("source_stage02_run_id") != STAGE02_RUN_ID:
        raise ValueError(f"backup refused: manifest source_stage02_run_id != {STAGE02_RUN_ID}")
    decision_record = json.loads((run_dir / "v2sa_decision_record.json").read_text(encoding="utf-8"))
    service = get_drive_service()
    drive_path_parts = V2SA_DRIVE_RESULT_PATH_PARTS + [run_id]
    drive_folder_id = ensure_drive_path(service, drive_path_parts)
    backup_manifest_path = run_dir / "drive_backup_manifest.json"
    local_files = sorted(path for path in run_dir.rglob("*") if path.is_file() and path.name != backup_manifest_path.name)
    uploads = []
    for path in local_files:
        uploaded = upload_or_update_drive_file(service, drive_folder_id, path)
        uploaded["bytes"] = path.stat().st_size
        uploads.append(uploaded)
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": run_id,
        "stage_run_id": run_id,
        "interpretation": decision_record.get("interpretation"),
        "official_validation_scoring_events": decision_record.get("official_validation_scoring_events"),
        "readout_complete": decision_record.get("readout_complete"),
        "merged_into_predeclared_readout": decision_record.get("merged_into_predeclared_readout"),
        "local_output_dir": str(run_dir),
        "drive_path": "My Drive/" + "/".join(drive_path_parts),
        "drive_path_parts": drive_path_parts,
        "drive_folder_id": drive_folder_id,
        "uploaded_file_names": [upload["name"] for upload in uploads],
        "uploaded_files": uploads,
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "holdout_test_contact": run_manifest.get("holdout_test_contact"),
        "official_validation_for_selection": run_manifest.get("official_validation_for_selection"),
        "no_final_model_selected": run_manifest.get("no_final_model_selected"),
    }
    backup_manifest["uploaded_files"] = backup_manifest["uploaded_files"] + [
        {"name": backup_manifest_path.name, "bytes": None, "self_reference": True}
    ]
    backup_manifest["uploaded_file_names"].append(backup_manifest_path.name)
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)
    print("stage_run_id:", backup_manifest["stage_run_id"])
    print("drive_path:", backup_manifest["drive_path"])
    print("drive_folder_id:", backup_manifest["drive_folder_id"])
    return backup_manifest

if RUN_V2SA_DRIVE_BACKUP and RUN_V2SA:
    if "result" not in globals():
        raise RuntimeError("RUN_V2SA_DRIVE_BACKUP=True requires running the addendum first.")
    v2sa_drive_backup_manifest = backup_v2sa_results_to_drive(result.output_dir)
else:
    print("RUN_V2SA_DRIVE_BACKUP is disabled or RUN_V2SA=False; no result backup uploaded.")

## Readout Display (Descriptive Dispersion Only)

In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd

    if "result" not in globals():
        raise RuntimeError("RUN_ARTIFACT_DISPLAY=True requires running the addendum first.")
    run_dir = Path(result.output_dir)
    decision_record_path = run_dir / "v2sa_decision_record.json"
    require_path(decision_record_path)
    decision_record = json.loads(decision_record_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "interpretation": decision_record.get("interpretation"),
        "seed_rule": decision_record.get("seed_rule"),
        "addendum_seeds": decision_record.get("addendum_seeds"),
        "official_validation_scoring_events": decision_record.get("official_validation_scoring_events"),
        "readout_complete": decision_record.get("readout_complete"),
        "merged_into_predeclared_readout": decision_record.get("merged_into_predeclared_readout"),
        "canonical_readout_unchanged": decision_record.get("canonical_readout_unchanged"),
        "no_final_model_selected": decision_record.get("no_final_model_selected"),
    }, indent=2))
    for display_name in ["v2sa_seed_dispersion_summary.csv", "v2sa_validation_readout.csv", "v2sa_same_row_baselines.csv", "v2sa_per_ticker_readout.csv", "v2sa_budget_ledger_row.csv"]:
        display_path = run_dir / display_name
        if display_path.exists():
            display(pd.read_csv(display_path))
        else:
            print(f"{display_name} not present (incomplete run).")
else:
    print("RUN_ARTIFACT_DISPLAY=False; no artifacts displayed.")

## Honest Interpretation Template (Predeclared)

Allowed wording: `disclosed post-hoc seed addendum`, `descriptive dispersion evidence
only`, `the addendum seeds show a delta range of [min, max] vs the same-row stratified
dummy`, `k of 6 addendum seeds have a positive delta`.

Required framing for any summary based on this run:

- The predeclared n=2 readout (seeds 101/202) remains the paper's canonical
  official-validation number. The addendum is reported separately and is NEVER merged,
  averaged, or pooled with it.
- All six addendum seeds are reported regardless of outcome. A wide or
  negative-touching spread is reported honestly; that outcome would strengthen the
  paper's stated n=2 limitation (Section 9), not weaken the paper.
- The addendum performs no selection and judges no pass/fail criteria; min/median/max
  and the positive-delta seed count are descriptive dispersion evidence only.
- Each of the six scoring events is a logged official-validation contact
  (`contact_type=official_validation_seed_addendum`, `for_selection=false`); append the
  emitted `v2sa_budget_ledger_row.csv` to the route budget accounting.
- If a prior partial run was lost before its checkpoints reached Drive, disclose it here
  (date, seeds started) and carry the disclosure into the budget accounting.

Forbidden: `final model`, `best`, `superior`, `outperforms`, `significant`,
`profitable`, `well-calibrated`, `clean test`, `out-of-sample proof`, any claim that the
addendum `confirms` or `replaces` the predeclared readout, and any model selection.